# Backpropagation on a two-layer network

The lesson derived the gradients of a two-layer network and checked them on one example. Here you
write the whole thing for a small regression set: the forward pass, the loss, the backward pass, a
numerical check that every gradient is right, and the gradient descent loop that uses them. By the
end the network fits a curve that no straight line could.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

A greenhouse keeper tried 40 thermostat settings, one per tray of seedlings, and measured how many
centimeters each tray grew in a week. The setting is rescaled so that the coldest tray is $-1$ and
the warmest is $1$. Growth peaks a little above the middle and falls off on both sides.
`init_params` builds random weights for a network with one input, a hidden layer of ReLU units and
one output. Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 40
x = np.sort(rng.uniform(-1, 1, n))                                   # thermostat setting, rescaled
growth = 1.5 * np.exp(-3 * (x - 0.2) ** 2) + 0.3 + rng.normal(0, 0.1, n)

X = x[:, None]                                                        # (40, 1): one row per tray
Y = growth[:, None]                                                   # (40, 1): one target per row


def relu(Z):
    return np.maximum(0.0, Z)


def init_params(n_hidden, seed=1):
    """Random weights for a 1 -> n_hidden -> 1 network, as [(W1, b1), (W2, b2)]."""
    r = np.random.default_rng(seed)
    W1 = r.normal(0, 1.0, (1, n_hidden))
    b1 = r.uniform(-1, 1, n_hidden)
    W2 = r.normal(0, 1.0 / np.sqrt(n_hidden), (n_hidden, 1))
    b2 = np.zeros(1)
    return [(W1, b1), (W2, b2)]


plt.scatter(x, growth, s=18)
plt.xlabel("thermostat setting (rescaled)")
plt.ylabel("growth in a week (cm)")
plt.show()
print(X.shape, Y.shape)

## Part 1 (coding task): The forward pass

The network has one input, a hidden layer of ReLU units, and one linear output. It is the lesson's
`forward` with two layers and no softmax, because the output is a number, not a class. With
examples as rows, each layer is `Z = A @ W + b`, so `W1` has shape `(1, n_hidden)` and `W2` has
shape `(n_hidden, 1)`.

Write `forward(X, params)`. It returns `(yhat, cache)`, where `yhat` has shape `(N, 1)` and
`cache` is a dictionary holding `"A0"` (the input), `"Z1"` and `"A1"`, the three arrays the
backward pass will need.

In [ ]:
def forward(X, params):
    """Predictions (N, 1) and the cache {"A0", "Z1", "A1"} for a 1 -> n_hidden -> 1 ReLU network."""
    # TODO
    raise NotImplementedError

In [ ]:
# A network small enough to run by hand: two hidden units, and one tray with input 2.
tiny = [(np.array([[1.0, -1.0]]), np.array([0.0, 0.5])),
        (np.array([[1.0], [2.0]]), np.array([0.1]))]
out, c = forward(np.array([[2.0]]), tiny)
assert isinstance(c, dict) and {"A0", "Z1", "A1"} <= set(c), "the cache should be a dict with keys A0, Z1 and A1"
assert np.allclose(c["Z1"], [[2.0, -1.5]]), f"Z1 should be [[2, -1.5]], got {c['Z1']}"
assert np.allclose(c["A1"], [[2.0, 0.0]]), f"A1 should be [[2, 0]], since ReLU turns the -1.5 off, got {c['A1']}"
assert np.allclose(out, [[2.1]]), f"the output should be 2 * 1 + 0 * 2 + 0.1 = 2.1, got {out}"

params = init_params(16)
yhat, cache = forward(X, params)
assert yhat.shape == (40, 1), f"40 trays should give shape (40, 1), got {yhat.shape}"
assert cache["Z1"].shape == (40, 16) and cache["A1"].shape == (40, 16), "Z1 and A1 should be (40, 16)"
assert np.array_equal(cache["A0"], X), "A0 should be the input"
assert np.all(cache["A1"] >= 0), "A1 should have no negative entries after ReLU"
print("Part 1 passed")

## Part 2 (coding task): The loss and its slope

The loss is the mean squared error over the $N$ trays, $L = \frac{1}{N}\sum_n (\hat y_n - y_n)^2$.
Backpropagation starts from the slope of the loss with respect to each prediction, and
differentiating one term gives $\partial L / \partial \hat y_n = 2(\hat y_n - y_n) / N$. The
lesson's output error for softmax and cross-entropy was $\hat{\mathbf{y}} - \mathbf{y}$ divided by
$N$. This is the same idea with a different loss, and the factor of 2 is the only new thing.

Write `mse_and_grad(yhat, Y)`. It returns the loss as a Python float and the array of slopes,
which has the same shape as `yhat`.

In [ ]:
def mse_and_grad(yhat, Y):
    """Mean squared error (a float) and dL/dyhat, an array shaped like yhat."""
    # TODO
    raise NotImplementedError

In [ ]:
loss, dyhat = mse_and_grad(np.array([[2.1], [0.5]]), np.array([[1.0], [1.0]]))
assert isinstance(loss, float), f"the loss should be a Python float, got {type(loss).__name__}"
assert abs(loss - 0.73) < 1e-9, f"(1.1^2 + 0.5^2) / 2 = 0.73, got {loss}"
assert dyhat.shape == (2, 1), f"the slopes should have shape (2, 1), got {dyhat.shape}"
assert np.allclose(dyhat, [[1.1], [-0.5]]), f"2 * (yhat - y) / N should be [[1.1], [-0.5]], got {dyhat.ravel()}"
loss0, _ = mse_and_grad(forward(X, params)[0], Y)
print(f"loss of the untrained network: {loss0:.4f}")
print("Part 2 passed")

## Part 3 (coding task): The backward pass

Now the lesson's four lines, in the rows-are-examples layout. The output error is the slope from
Part 2, $\boldsymbol{\delta}^{(2)} = \partial L / \partial \hat{\mathbf{y}}$, because the output
layer has no activation. Each layer's gradients are outer products summed over the batch,
`A_prev.T @ delta` for the weights and `delta.sum(axis=0)` for the bias. To move the error back
through the hidden layer, multiply by `W2.T` and then by ReLU's slope, which is 1 where `Z1 > 0`
and 0 elsewhere.

Write `backward(cache, params, dyhat)`. It returns `[(dW1, db1), (dW2, db2)]`, each gradient with
the same shape as its parameter.

In [ ]:
def backward(cache, params, dyhat):
    """Gradients [(dW1, db1), (dW2, db2)] of the loss, given the cache and dL/dyhat."""
    # TODO
    raise NotImplementedError

In [ ]:
# The tiny network again, with target 1: the output 2.1 is off by 1.1, so dL/dyhat = 2.2.
out, c = forward(np.array([[2.0]]), tiny)
_, dy = mse_and_grad(out, np.array([[1.0]]))
(dW1, db1), (dW2, db2) = backward(c, tiny, dy)
assert dW1.shape == (1, 2) and db1.shape == (2,) and dW2.shape == (2, 1) and db2.shape == (1,), "each gradient should have the shape of its parameter"
assert np.allclose(db2, [2.2]), f"db2 is the output error 2 * (2.1 - 1) = 2.2, got {db2}"
assert np.allclose(dW2, [[4.4], [0.0]]), f"dW2 = A1^T delta2 should be [[4.4], [0]], got {dW2.ravel()}"
assert np.allclose(db1, [2.2, 0.0]), f"the hidden error should be [2.2, 0], since the second unit was off, got {db1}"
assert np.allclose(dW1, [[4.4, 0.0]]), f"dW1 = x * delta1 should be [[4.4, 0]], got {dW1}"

yhat, cache = forward(X, params)
_, dy = mse_and_grad(yhat, Y)
grads = backward(cache, params, dy)
for (W, b), (dW, db) in zip(params, grads):
    assert dW.shape == W.shape and db.shape == b.shape, "on the real data too, every gradient has its parameter's shape"
print("Part 3 passed")

## Part 4 (coding task): Check the gradients numerically

A gradient derived by hand can be wrong in a way that still runs. The check is the central
difference from the lesson. Nudge one parameter entry up by $h$ and down by $h$, run the forward
pass and the loss both times, and divide the change in the loss by $2h$. That is the slope of the
loss along that one entry, computed with no calculus at all. Doing it for every entry costs two
forward passes each, far too slow for training and perfect for a test.

Write `numeric_grads(params, X, Y, h=1e-5)`. It returns gradients in the same
`[(dW1, db1), (dW2, db2)]` form as `backward`, every entry by central difference. `np.ndindex(arr.shape)`
walks every index of an array. Put each entry back to its old value after nudging it.

In [ ]:
def numeric_grads(params, X, Y, h=1e-5):
    """[(dW1, db1), (dW2, db2)] by central differences, one entry at a time."""
    # TODO
    raise NotImplementedError

In [ ]:
numeric = numeric_grads(params, X, Y)
assert len(numeric) == 2 and all(len(pair) == 2 for pair in numeric), "return [(dW1, db1), (dW2, db2)]"
gaps = []
for (dW, db), (nW, nb), (W, b) in zip(grads, numeric, params):
    assert nW.shape == W.shape and nb.shape == b.shape, "each numeric gradient has its parameter's shape"
    gaps += [np.abs(nW - dW).max(), np.abs(nb - db).max()]
    assert np.abs(nW - dW).max() < 1e-6, f"dW differs from the numeric slope by {np.abs(nW - dW).max():.2e}"
    assert np.abs(nb - db).max() < 1e-6, f"db differs from the numeric slope by {np.abs(nb - db).max():.2e}"
# A real central difference depends on h; a copy of backward does not.
coarse = numeric_grads(params, X, Y, h=1e-2)
spread = max(np.abs(nc - nf).max() for (nc, _), (nf, _) in zip(coarse, numeric))
assert spread > 0.0, f"the numeric slopes must move when h changes; got spread {spread:.1e}"
assert np.allclose(forward(X, params)[0], yhat), "the parameters should be unchanged after the check"
print(f"largest gap between backward and the numeric slopes: {max(gaps):.1e}")
print("Part 4 passed")

## Part 5 (coding task): One gradient descent step

A step of gradient descent subtracts the learning rate times each gradient from its parameter.
With correct gradients a small step must lower the loss, which is the whole point of computing
them.

Write `step(params, grads, lr)`. It returns a new parameter list `[(W1, b1), (W2, b2)]` with every
array moved by `-lr` times its gradient. Do not modify the arrays in `params`. Build new ones.

In [ ]:
def step(params, grads, lr):
    """New parameters after one gradient descent step. Leaves `params` unchanged."""
    # TODO
    raise NotImplementedError

In [ ]:
before = [(W.copy(), b.copy()) for W, b in params]
new_params = step(params, grads, 0.1)
assert all(np.array_equal(W, W0) and np.array_equal(b, b0) for (W, b), (W0, b0) in zip(params, before)), "step() changed the old parameters in place"
assert np.allclose(new_params[1][1], params[1][1] - 0.1 * grads[1][1]), "b2 should move by -lr * db2"
assert np.allclose(new_params[0][0], params[0][0] - 0.1 * grads[0][0]), "W1 should move by -lr * dW1"
loss_before = mse_and_grad(forward(X, params)[0], Y)[0]
loss_after = mse_and_grad(forward(X, new_params)[0], Y)[0]
assert loss_after < loss_before, f"one step should lower the loss, but it went from {loss_before:.4f} to {loss_after:.4f}"
print(f"loss {loss_before:.4f} -> {loss_after:.4f} after one step")
print("Part 5 passed")

## Part 6 (coding task): The training loop

Everything is in place: forward pass, loss, backward pass, step. Training repeats them. Each
iteration runs the forward pass on all 40 trays, computes the loss and its slope, runs the
backward pass, and takes one step.

Write `train(params, X, Y, lr, steps)`. It returns `(params, losses)`, the parameters after
`steps` iterations and a list with the loss recorded before each step. Starting from fresh
parameters, 3,000 steps at a learning rate of 0.1 should take the loss from about 1.6 to about
0.01, which is the variance of the noise in the measurements. The check draws the loss curve and
the fitted curve.

In [ ]:
def train(params, X, Y, lr, steps):
    """Gradient descent for `steps` iterations. Returns (params, losses)."""
    # TODO
    raise NotImplementedError

In [ ]:
trained, losses = train(init_params(16), X, Y, lr=0.1, steps=3000)
assert len(losses) == 3000, f"one loss per step: expected 3000, got {len(losses)}"
assert np.isclose(losses[0], mse_and_grad(forward(X, init_params(16))[0], Y)[0]), \
    f"losses[0] must be the loss of the untrained network, got {losses[0]:.4f}"
assert losses[0] > 1.0, f"the first loss is the untrained network's, about 1.6, got {losses[0]:.4f}"
assert losses[-1] < 0.02, f"after 3000 steps the loss should be near 0.01, got {losses[-1]:.4f}"
assert losses[-1] < losses[100] < losses[0], "the loss should keep falling"

grid = np.linspace(-1, 1, 200)[:, None]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(losses)
ax1.set_yscale("log")
ax1.set_xlabel("step")
ax1.set_ylabel("mean squared error")
ax2.scatter(x, growth, s=18, label="trays")
ax2.plot(grid, forward(grid, init_params(16))[0], "--", label="before training")
ax2.plot(grid, forward(grid, trained)[0], label="after training")
ax2.set_xlabel("thermostat setting (rescaled)")
ax2.set_ylabel("growth (cm)")
ax2.legend()
plt.show()
print(f"loss {losses[0]:.4f} -> {losses[-1]:.4f}")
print("Part 6 passed")

## Solution

### Part 1: The forward pass

In [ ]:
def forward(X, params):
    """Predictions (N, 1) and the cache {"A0", "Z1", "A1"} for a 1 -> n_hidden -> 1 ReLU network."""
    (W1, b1), (W2, b2) = params
    Z1 = X @ W1 + b1                                   # (N, n_hidden)
    A1 = relu(Z1)
    yhat = A1 @ W2 + b2                                # (N, 1)
    return yhat, {"A0": X, "Z1": Z1, "A1": A1}

### Part 2: The loss and its slope

In [ ]:
def mse_and_grad(yhat, Y):
    """Mean squared error (a float) and dL/dyhat, an array shaped like yhat."""
    diff = yhat - Y
    loss = float(np.mean(diff ** 2))
    return loss, 2 * diff / len(Y)

### Part 3: The backward pass

In [ ]:
def backward(cache, params, dyhat):
    """Gradients [(dW1, db1), (dW2, db2)] of the loss, given the cache and dL/dyhat."""
    (W1, b1), (W2, b2) = params
    delta2 = dyhat                                     # output error: no activation at the output
    dW2 = cache["A1"].T @ delta2                       # outer products summed over the batch
    db2 = delta2.sum(axis=0)
    delta1 = (delta2 @ W2.T) * (cache["Z1"] > 0)       # back through W2, then ReLU's slope
    dW1 = cache["A0"].T @ delta1
    db1 = delta1.sum(axis=0)
    return [(dW1, db1), (dW2, db2)]

### Part 4: Check the gradients numerically

In [ ]:
def numeric_grads(params, X, Y, h=1e-5):
    """[(dW1, db1), (dW2, db2)] by central differences, one entry at a time."""
    def loss_now():
        return mse_and_grad(forward(X, params)[0], Y)[0]

    grads = []
    for W, b in params:
        pair = []
        for arr in (W, b):
            d = np.zeros_like(arr)
            for idx in np.ndindex(arr.shape):
                old = arr[idx]
                arr[idx] = old + h
                up = loss_now()
                arr[idx] = old - h
                down = loss_now()
                arr[idx] = old
                d[idx] = (up - down) / (2 * h)
            pair.append(d)
        grads.append(tuple(pair))
    return grads

### Part 5: One gradient descent step

In [ ]:
def step(params, grads, lr):
    """New parameters after one gradient descent step. Leaves `params` unchanged."""
    return [(W - lr * dW, b - lr * db) for (W, b), (dW, db) in zip(params, grads)]

### Part 6: The training loop

In [ ]:
def train(params, X, Y, lr, steps):
    """Gradient descent for `steps` iterations. Returns (params, losses)."""
    losses = []
    for _ in range(steps):
        yhat, cache = forward(X, params)
        loss, dyhat = mse_and_grad(yhat, Y)
        losses.append(loss)
        grads = backward(cache, params, dyhat)
        params = step(params, grads, lr)
    return params, losses